# فصل ۱۴: دانش خارجی و معماری RAG (External Knowledge & RAG)
## بخش ۲: معماری RAG - اپیزود دوم: استخراج منابع و ارجاعات (Source Documents & Citations)

### 🕳️ معضل جعبه سیاه (The Black Box Dilemma) در سیستم‌های RAG
در درسنامه قبل، اولین خط لوله RAG خود را ساختیم. اما یک چالش بزرگ و ساختاری وجود داشت:
سیستم ما مانند یک **جعبه سیاه (Black Box)** رفتار می‌کرد! مدل جوابی تولید می‌کرد، اما کاربر نمی‌دانست این جواب از کجا آمده است.

در سیستم‌های حساس و سازمانی (نظیر دستیارهای حقوقی، پزشکی، مهندسی و مالی)، **پاسخ بدون رفرنس و منبع فاقد ارزش است**:
> **مهندس می‌پرسد:** *«حداکثر فشار مجاز این مخزن چقدر است؟»*
> **مدل پاسخ می‌دهد:** *«۱۵ بار.»*
> **مهندس بلافاصله می‌پرسد:** *«مطمئنی؟ طبق کدام سند و کدام صفحه؟»*

یک سیستم RAG تجاری و قابل اعتماد باید بتواند همراه با جواب بگوید:
> *«فشار مجاز ۱۵ بار است. (منبع: دستورالعمل ایمنی سال ۱۴۰۲، صفحه ۱۲)»*

در این درسنامه یاد می‌گیریم که چگونه معماری زنجیره را با **`RunnableParallel`** و متد **`.assign()`** ارتقا دهیم تا علاوه بر متن پاسخ، **اسناد مرجع (`Source Documents`)** نیز در خروجی حفظ و تحویل داده شوند.

### 🔍 چرا زنجیره قبلی منابع را دور می‌ریخت؟

در خط لوله ساده قبلی مسیر داده‌ها به این شکل بود:
$$\text{Question} \longrightarrow \text{Retriever} \longrightarrow \text{Format Docs} \longrightarrow \text{Prompt} \longrightarrow \text{LLM} \longrightarrow \text{String Output}$$

مشکل این بود که اسناد بازیابی‌شده در میانه راه (توسط تابع `format_docs`) به یک متن ساده تبدیل شده و وارد پرامپت می‌شدند، و در مرحله خروجی فقط یک رشته متنی خام از مدل دریافت می‌کردیم؛ تمام اشیاء `Document` و متادیتای آن‌ها (نام فایل، شماره صفحه، تاریخ) گم می‌شدند!

**راه‌حل معماری:** استفاده از دو شاخه موازی با `RunnableParallel` جهت نگهداری اسناد خام در یک مسیر و ارسال متن فرمت‌شده به مدل در مسیر دیگر.

### 🔄 دیاگرام معماری موازی با `RunnableParallel` و `.assign()`:

```mermaid
flowchart TD
    Q["👤 سوال کاربر (Input Question)"] --> RP["🔀 RunnableParallel"]
    RP --> |شاخه اول| C["📄 context: اسناد خام Document"]
    RP --> |شاخه دوم| QK["❓ question: متن سوال"]
    C --> ASSIGN["➕ .assign(answer=...)"]
    QK --> ASSIGN
    ASSIGN --> FD["📝 format_docs(context)"]
    FD --> PR["💬 Prompt Template"]
    PR --> LLM["🤖 Google Gemini LLM"]
    LLM --> OP["🔤 StrOutputParser"]
    OP --> OUT["📦 خروجی نهایی: دیکشنری شامل answer + context + question"]
```

### ۱. آماده‌سازی اسناد سازمانی و پایگاه برداری (Chroma Vector DB)

برای شبیه‌سازی دقیق درسنامه، اسناد واقعی سازمان شامل قوانین گارانتی و دستورالعمل ایمنی را به همراه متادیتای `source` و `page` در Chroma ذخیره می‌کنیم:

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

# فعال‌سازی حالت آفلاین برای مدل تعبیه محلی
os.environ["HF_HUB_OFFLINE"] = "1"

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# مدل امبدینگ چندزبانه سریع و دقیق
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# اسناد تجاری و فنی با شناسنامه کامل (Metadata)
sample_docs = [
    Document(
        page_content="تمامی محصولات شرکت مشمول ۱۸ ماه گارانتی تعویض بی‌قیدوشرط قطعات و ۵ سال خدمات پس از فروش می‌باشند.",
        metadata={"source": "warranty_policy_2024.pdf", "page": 3, "category": "warranty"}
    ),
    Document(
        page_content="خدمات پس از فروش به مدت ۵ سال تضمین می‌گردد و شامل تأمین کلیه قطعات یدکی استاندارد است.",
        metadata={"source": "service_manual.txt", "page": 1, "category": "service"}
    ),
    Document(
        page_content="فشار مجاز کاری مخزن تحت فشار بخار حداکثر ۱۵ بار (Bar) تعیین شده و شیر اطمینان روی ۱۶ بار کالیبره می‌شود.",
        metadata={"source": "safety_instruction_1402.pdf", "page": 12, "category": "engineering"}
    )
]

# راه‌اندازی پایگاه برداری Chroma
vector_db = Chroma.from_documents(
    documents=sample_docs,
    embedding=embedding_model,
    collection_name="company_knowledge"
)

# تعریف موتور بازیاب اسناد (همیشه ۲ سند برتر مرتبط را برگردان)
retriever = vector_db.as_retriever(search_kwargs={"k": 2})
print("✅ دیتابیس برداری و بازیاب اسناد آماده شدند.")

Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2569.07it/s]


✅ دیتابیس برداری و بازیاب اسناد آماده شدند.


### ۲. راه‌اندازی مدل زبانی جمنای و پرامپت مقیدساز

مدل زبانی را با **`gemini-flash-lite-latest`** و دمای صفر (`temperature=0`) راه‌اندازی می‌کنیم تا استنتاج کاملاً بر متن کانتکست متکی باشد:

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# راه‌اندازی مدل زبانی جمنای با کلید فعال پروژه
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    api_key=os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY"),
    temperature=0
)

# طراحی الگوی پرامپت مقیدساز RAG
template = """Answer the question based only on the following context:
{context}

Question: {question}
If you don't know the answer based on the context, simply say "I don't know". Do not make up an answer.
"""

prompt = ChatPromptTemplate.from_template(template)
print(f"✅ مدل زبانی '{llm.model}' و پرامپت با موفقیت آماده شدند.")

✅ مدل زبانی 'gemini-flash-lite-latest' و پرامپت با موفقیت آماده شدند.


### ۳. ساخت زنجیره موازی RAG با قابلیت بازگشت منبع (`rag_chain_with_source`)

اینجا نبوغ LCEL را مشاهده می‌کنید. ما از ترکیب **`RunnableParallel`** و **`RunnablePassthrough.assign`** استفاده می‌کنیم:

```python
rag_chain_with_source = (
    RunnableParallel(
        {"context": retriever, "question": RunnablePassthrough()}
    )
    .assign(answer=(
        RunnablePassthrough.assign(
            context=lambda x: format_docs(x["context"])
        )
        | prompt
        | llm
        | StrOutputParser()
    ))
)
```

**گام‌به‌گام اتفاقاتی که رخ می‌دهد:**
1. `RunnableParallel` ورودی را گرفته و به صورت همزمان:
   - سوال را در کلید `question` نگه می‌دارد.
   - اسناد خام بازیابی‌شده (`List[Document]`) را در کلید `context` ذخیره می‌کند.
2. متد `.assign(answer=...)` اطلاعات قبلی (اسناد خام و سوال) را حفظ می‌کند و یک کلید جدید به نام `answer` به دیکشنری اضافه می‌کند.
3. در درون `assign`، اسناد خام با تابع `format_docs` به متن تبدیل شده، وارد پرامپت می‌شوند و مدل جمنای پاسخ را می‌نویسد.

In [3]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    """تبدیل لیست اسناد بازیابی شده به یک رشته متنی یکپارچه برای پرامپت"""
    return "\n\n".join([d.page_content for d in docs])

# ساخت معماری پیشرفته RAG با خروجی ساختاریافته (پاسخ + منابع)
rag_chain_with_source = (
    RunnableParallel(
        {"context": retriever, "question": RunnablePassthrough()}
    )
    .assign(answer=(
        RunnablePassthrough.assign(
            context=lambda x: format_docs(x["context"])
        )
        | prompt
        | llm
        | StrOutputParser()
    ))
)

print("✅ زنجیره RAG با قابلیت حفظ منبع کامپایل شد.")

✅ زنجیره RAG با قابلیت حفظ منبع کامپایل شد.


### ۴. تست عملی سناریوی اول: شرایط گارانتی محصولات

اکنون سوال *«شرایط گارانتی محصولات چیست؟»* را اجرا می‌کنیم و خروجی غنی دیکشنری را بررسی می‌کنیم:

In [4]:
query = "شرایط گارانتی محصولات چیست؟"
response = rag_chain_with_source.invoke(query)

print("--- AI Answer ---")
print(response['answer'])

print("\n--- Sources Used ---")
for i, doc in enumerate(response['context']):
    source_name = doc.metadata.get('source', 'Unknown Source')
    page_num = doc.metadata.get('page', 'N/A')
    print(f"Source {i+1}: {source_name} (Page {page_num})")
    print(f"Snippet: {doc.page_content[:100]}...")
    print("-" * 30)

--- AI Answer ---
بر اساس متن داده شده، شرایط گارانتی محصولات شامل ۱۸ ماه گارانتی تعویض بی‌قیدوشرط قطعات است. همچنین خدمات پس از فروش به مدت ۵ سال تضمین می‌گردد و شامل تأمین کلیه قطعات یدکی استاندارد است.

--- Sources Used ---
Source 1: warranty_policy_2024.pdf (Page 3)
Snippet: تمامی محصولات شرکت مشمول ۱۸ ماه گارانتی تعویض بی‌قیدوشرط قطعات و ۵ سال خدمات پس از فروش می‌باشند....
------------------------------
Source 2: service_manual.txt (Page 1)
Snippet: خدمات پس از فروش به مدت ۵ سال تضمین می‌گردد و شامل تأمین کلیه قطعات یدکی استاندارد است....
------------------------------


### ۵. تست عملی سناریوی دوم: سوال حساس مهندسی (فشار مجاز مخزن)

پاسخ به سوال مهندس فنی: استناد صریح به دستورالعمل سال ۱۴۰۲ و صفحه ۱۲:

In [5]:
query_tech = "فشار مجاز کاری مخزن چقدر تعیین شده است؟"
resp_tech = rag_chain_with_source.invoke(query_tech)

print("🔧 پاسخ فنی مدل:")
print(resp_tech['answer'])

print("\n📑 سند و ارجاع دقیق استخراج‌شده:")
for idx, d in enumerate(resp_tech['context'], 1):
    print(f"{idx}. فایل: {d.metadata.get('source')} | صفحه: {d.metadata.get('page')}")
    print(f"   متن سند: {d.page_content}")

🔧 پاسخ فنی مدل:
بر اساس متن، فشار مجاز کاری مخزن حداکثر ۱۵ بار (Bar) تعیین شده است.

📑 سند و ارجاع دقیق استخراج‌شده:
1. فایل: safety_instruction_1402.pdf | صفحه: 12
   متن سند: فشار مجاز کاری مخزن تحت فشار بخار حداکثر ۱۵ بار (Bar) تعیین شده و شیر اطمینان روی ۱۶ بار کالیبره می‌شود.
2. فایل: warranty_policy_2024.pdf | صفحه: 3
   متن سند: تمامی محصولات شرکت مشمول ۱۸ ماه گارانتی تعویض بی‌قیدوشرط قطعات و ۵ سال خدمات پس از فروش می‌باشند.


### ۶. مزیت شگفت‌انگیز در خطایابی سیستم (Debugging & Observability)

گاهی اوقات مدل پاسخی غیرمنتظره یا ناقص می‌دهد. در معماری ساده قبلی نمی‌دانستیم مقصر کیست:
- **آیا Retriever مقصر بوده است؟** (سند اشتباه یا نامربوط آورده؟)
- **یا LLM مقصر بوده است؟** (سند درست بوده اما مدل نتوانسته آن را درک کند؟)

با در اختیار داشتن کلید `response['context']`، شفافیت ۱۰۰٪ فراهم است و مستقیماً می‌توانیم بررسی کنیم دقیقاً چه متونی در زمان پرسش به پرامپت تزریق شده‌اند.

In [6]:
# بررسی صحت عملکرد بازیاب با بازرسی مستقیم کلید context
retrieved_docs_count = len(response['context'])
retrieved_sources = [doc.metadata.get('source') for doc in response['context']]

print(f"تعداد اسناد ارائه‌شده به پرامپت مدل: {retrieved_docs_count}")
print(f"لیست فایل‌های منبع استناد: {retrieved_sources}")

تعداد اسناد ارائه‌شده به پرامپت مدل: 2
لیست فایل‌های منبع استناد: ['warranty_policy_2024.pdf', 'service_manual.txt']


### 🎯 جمع‌بندی درسنامه RAG اپیزود دوم:

1. **پایان دوران جعبه سیاه:** ارتقای سیستم RAG از خروجی متنی صرف (`String Output`) به خروجی ساختاریافته شامل پاسخ و اسناد خام مرجع.
2. **قدرت `RunnableParallel`:** تفکیک مسیر پردازش به دو شاخه موازی (نگهداری اسناد برای خروجی + تبدیل اسناد به متن برای پرامپت).
3. **متد `.assign()`:** الصاق کلید `answer` به دیکشنری بدون حذف داده‌های بازیابی‌شده قبلی.
4. **استخراج متادیتا در فرانت‌اند:** توانایی نمایش نام فایل، شماره صفحه، لینک و قطعه متن (Snippet) به کاربر جهت افزایش اعتمادپذیری سیستم.
5. **پیش‌درآمد درس بعدی:** رفع مشکل شباهت بیش از حد و تکراری بودن اسناد بازیابی‌شده با تنظیم و تیونینگ موتور بازیاب (Retriever Tuning).